# Agent Loop Demo (No Memory) — LangGraph + Azure OpenAI

Same demo as before — the 5-step agent loop, no memory — but now built with **LangGraph** instead of a plain Python `while` loop.

**Same 5 steps, mapped onto a graph:**

| Loop step | LangGraph piece |
|---|---|
| Perceive | a **node** |
| Plan | a **node** |
| Act | a **node** |
| Observe | a **node** |
| Reflect | a **node** that decides: loop back to Perceive, or **stop** (`END`) |

The "loop back or stop" decision becomes a **conditional edge** in LangGraph — that's the main new concept here. Everything else is the exact same logic as the plain-Python version.

**Example task (unchanged):** *"If the stock price drops below Rs 100, send an alert."* Price is simulated with a random number, same as before.


## 0. Setup

In [ ]:
# pip install langgraph langchain-openai python-dotenv

import os
import random
from typing import TypedDict
from dotenv import load_dotenv

from langchain_openai import AzureChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langgraph.graph import StateGraph, END

load_dotenv()

llm = AzureChatOpenAI(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    azure_deployment=os.environ["AZURE_OPENAI_DEPLOYMENT"],
    api_version=os.environ.get("AZURE_OPENAI_API_VERSION", "2024-10-21"),
    temperature=0,
)

PRICE_THRESHOLD = 100.0
print("Setup done. Threshold price: Rs", PRICE_THRESHOLD)


## 1. Define the State

In LangGraph, the "state" is just a dictionary that gets passed from node to node. Each node reads from it and returns updates to it.

Think of this as the whiteboard all 5 steps share while the loop is running (this is NOT memory across loop *runs* — it resets every time you call the graph, same as our no-memory version before).

In [ ]:
class AgentState(TypedDict):
    price: float
    decision: str
    action_result: str


## 2. Define the 5 nodes
Same logic as before — just written as functions that take `state` in and return updates to `state`.

In [ ]:
def perceive_node(state: AgentState) -> AgentState:
    """PERCEIVE: look at what's happening right now."""
    current_price = round(random.uniform(80, 120), 2)   # simulated price
    print(f"[PERCEIVE] Current price: Rs {current_price}")
    return {"price": current_price}


In [ ]:
plan_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are the planning step of a simple agent. "
     "You are told the current stock price and a threshold. "
     "If the price is BELOW the threshold, reply with exactly one word: SEND_EMAIL. "
     "Otherwise, reply with exactly one word: WAIT. "
     "Do not explain. Only output that one word."),
    ("user", "Current price: Rs {price}. Threshold: Rs {threshold}."),
])
plan_chain = plan_prompt | llm | StrOutputParser()

def plan_node(state: AgentState) -> AgentState:
    """PLAN: decide what to do next."""
    decision = plan_chain.invoke({"price": state["price"], "threshold": PRICE_THRESHOLD}).strip()
    print(f"[PLAN] Decision: {decision}")
    return {"decision": decision}


In [ ]:
def act_node(state: AgentState) -> AgentState:
    """ACT: do it."""
    if state["decision"] == "SEND_EMAIL":
        # --- in a real version, send an actual email here, e.g. via smtplib ---
        print(f"[ACT] 📧 Sending email: 'Alert! Price dropped to Rs {state['price']}'")
        return {"action_result": "email_sent"}
    else:
        print("[ACT] Nothing to do — price is fine.")
        return {"action_result": "no_action"}


In [ ]:
def observe_node(state: AgentState) -> AgentState:
    """OBSERVE: see what came back."""
    print(f"[OBSERVE] Result of action: {state['action_result']}")
    return {}


In [ ]:
def reflect_node(state: AgentState) -> AgentState:
    """REFLECT: just prints — the actual stop/loop decision is made by the
    conditional edge function below, using this same state."""
    if state["action_result"] == "email_sent":
        print("[REFLECT] Goal met (alert sent). -> STOP")
    else:
        print("[REFLECT] Goal not met yet. -> LOOP BACK to Perceive")
    return {}


## 3. The loop-back-or-stop decision

This is the one genuinely new idea in LangGraph: instead of an `if/break` inside a `while` loop, you write a small function that looks at the state and returns a **string naming which node to go to next**. LangGraph calls this a **conditional edge**.

In [ ]:
def decide_next_step(state: AgentState) -> str:
    """Reads the state and decides: go back to 'perceive', or 'stop'."""
    if state["action_result"] == "email_sent":
        return "stop"
    return "loop"


## 4. Build the graph
Wire the 5 nodes together in order, then attach the conditional edge after Reflect.

In [ ]:
graph = StateGraph(AgentState)

graph.add_node("perceive", perceive_node)
graph.add_node("plan", plan_node)
graph.add_node("act", act_node)
graph.add_node("observe", observe_node)
graph.add_node("reflect", reflect_node)

graph.set_entry_point("perceive")
graph.add_edge("perceive", "plan")
graph.add_edge("plan", "act")
graph.add_edge("act", "observe")
graph.add_edge("observe", "reflect")

# after Reflect: either go back to Perceive, or stop (END)
graph.add_conditional_edges(
    "reflect",
    decide_next_step,
    {
        "loop": "perceive",
        "stop": END,
    },
)

agent = graph.compile()
print("Graph compiled.")


## 5. Run it
Same result as the plain-Python `while` loop version — just expressed as a graph.

In [ ]:
final_state = agent.invoke(
    {"price": 0.0, "decision": "", "action_result": ""},
    config={"recursion_limit": 25},   # safety cap, similar to max_iterations before
)
print("\\nFinal state:", final_state)


## Plain Python loop vs. LangGraph — what actually changed

| | Plain Python `while` loop | LangGraph |
|---|---|---|
| The 5 steps | 5 functions called in order | 5 **nodes** called in order |
| Data passed between steps | function arguments / return values | a shared **state** dict |
| "Loop back or stop" | an `if/break` inside the loop | a **conditional edge** function |
| Running it | call `run_agent_loop()` | call `agent.invoke(...)` |

**Nothing about the underlying 5-step logic changed.** LangGraph doesn't add a 6th step or change what Perceive/Plan/Act/Observe/Reflect *mean* — it just gives you a standard structure (nodes + edges) for wiring the same loop together, which becomes more useful once you have branching paths, multiple agents, or want to visualize the flow.

Still no memory here, same as before — that's the next thing to add on top of this graph.
